In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

import cooler
import bioframe
import cooltools

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import h5py

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic'
# clusts = list(np.loadtxt(f'{projdir}/txt/liftover.txt', dtype=str))

# donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
# tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
# clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]
clusts = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc'

liftover = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
liftover.shape

In [ ]:
liftover.head()

In [ ]:
indir = f'/u/project/cluo_scratch/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic/merged_contacts/cooler/liftover'
s = clusts[0]
# res = 1000000
res = 100000
clr = cooler.Cooler(f'{indir}/{s}.mcool::/resolutions/{res}')

In [ ]:
clr.bins()[:].head()

In [ ]:
clr.bins()[:].shape

In [ ]:
clr.bins()[:]['weight'].isna().sum()

In [ ]:
# weights = pd.read_csv(f'{projdir}/csv/weights/liftover/{s}.1mb.csv', sep=',', header=None, index_col=None)
# final_weights = weights[0].astype(float)

# with h5py.File(f'{indir}/{s}.10kb.mcool', "r+") as f:
#     grp = f[f"/resolutions/{res}/bins"]
#     if "weight" in grp:
#         del grp["weight"] 
#     grp.create_dataset("weight", data=final_weights, dtype=np.float64)

In [ ]:
# for s in clusts:
#     clr = cooler.Cooler(f'{indir}/{s}.10kb.mcool::/resolutions/{res}')
#     weights = pd.read_csv(f'{projdir}/csv/weights/liftover/{s}.1mb.csv', sep=',', header=None, index_col=None)
#     final_weights = weights[0].astype(float)

#     with h5py.File(f'{indir}/{s}.10kb.mcool', "r+") as f:
#         grp = f[f"/resolutions/{res}/bins"]
#         if "weight" in grp:
#             del grp["weight"] 
#         grp.create_dataset("weight", data=final_weights, dtype=np.float64)

In [ ]:
# gc_path = '/u/project/cluo/terencew/reference/hg38/bed/hg38_1mb_gc.bed'
gc_path = '/u/project/cluo/terencew/reference/hg38/bed/hg38_100kb_gc.bed'
gc_cov = pd.read_csv(gc_path, sep='\t', header=0, index_col=0)
gc_cov['chrom'] = [f'chr{x}' for x in gc_cov['chrom']]
gc_cov.shape

In [ ]:
gc_cov.head()

In [ ]:
autosome = [f'chr{x}' for x in range(1,23)]

In [ ]:
# obtain first 3 eigenvectors
weight_col = 'weight'
cis_eigs = cooltools.eigs_cis(clr, gc_cov, clr_weight_name=weight_col, n_eigs=3)
eigs = cis_eigs[1]
eigs.shape

In [ ]:
eigs.head()

In [ ]:
def process_eigs(eigs, cutoff=0):
    eigs['E1_comp'] = 'A'
    mask = eigs['E1'] < cutoff 
    eigs.loc[mask, 'E1_comp'] = 'B'

    eigs['E2_comp'] = 'A'
    mask = eigs['E2'] < cutoff
    eigs.loc[mask, 'E2_comp'] = 'B'

    eigs['E3_comp'] = 'A'
    mask = eigs['E3'] < cutoff
    eigs.loc[mask, 'E3_comp'] = 'B'
    return eigs

In [ ]:
mask = [x in autosome for x in eigs['chrom']]
auto_eigs = eigs[mask].dropna()
auto_eigs = process_eigs(auto_eigs)
auto_eigs.shape

In [ ]:
auto_eigs.head()

In [ ]:
auto_eigs['E1_comp'].value_counts(), \
auto_eigs['E2_comp'].value_counts(), \
auto_eigs['E3_comp'].value_counts()

### process everything

In [ ]:
def get_comps(s, res=100000):
    indir = f'/u/project/cluo_scratch/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic/merged_contacts/cooler/liftover'
    clr = cooler.Cooler(f'{indir}/{s}.mcool::/resolutions/{res}')
    
    ###
    # obtain first 3 eigenvectors
    cis_eigs = cooltools.eigs_cis(clr,gc_cov, clr_weight_name=weight_col, n_eigs=3)
    eig_tracks = cis_eigs[0]
    eig_tracks['sample'] = s
    
    eigs = cis_eigs[1]
    mask = [x in autosome for x in eigs['chrom']]
    auto_eigs = eigs[mask].dropna()
    auto_eigs = process_eigs(auto_eigs)
    auto_eigs['sample'] = s
    return (eig_tracks, auto_eigs)

In [ ]:
%%time

with ProcessPoolExecutor(max_workers=10) as executor:
    results = list(tqdm(executor.map(get_comps, clusts), total=len(clusts)))

In [ ]:
tmp_tracks = [x[0] for x in results]
tmp_eigs = [x[1] for x in results]

In [ ]:
tracks = pd.concat(tmp_tracks).reset_index(drop=True)
tracks.shape

In [ ]:
tracks.head()

In [ ]:
comps = pd.concat(tmp_eigs).reset_index(drop=True)
comps.shape

In [ ]:
comps.head()

In [ ]:
tracks.to_csv(f'{projdir}/csv/compartments/cooltools/liftover/merged.tracks.100kb.csv', sep='\t')
comps.to_csv(f'{projdir}/csv/compartments/cooltools/liftover/merged.comps.100kb.csv', sep='\t')

In [ ]:
!date